In [1]:
import sys, os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_23_try_0.pickle

In [3]:
%%RecordEvent
%%time
### cell 24 ###

use_modin = os.environ.get("IREWR_WITH_MODIN") == "True"
countries = ["USA", "India"]

# 1) Fast boolean filter for us_ind (on original df, whether modin or pandas)
mask = df["first_country"].isin(countries)
us_ind = df.loc[mask]

# 2) If using Modin, convert just for the heavy groupby; leave df itself untouched
df_p = df._to_pandas() if use_modin else df

# 3) Compute the same cumulative cross‐country sums by pivoting on year first,
#    then cumsum across the country columns (axis=1) to match original .cumsum(axis=0).T
data_sub = (
    df_p
      .loc[df_p["first_country"].isin(countries), ["year_added", "first_country"]]
      .groupby(["year_added", "first_country"])  # group on year, then country
      .size()                                       # fast count
      .unstack(fill_value=0)                        # pivot into year×country
      .loc[:, countries]                            # ensure column order [USA, India]
      .cumsum(axis=1)                               # accumulate across country columns
      .astype(float)                                # match original float dtype
)


CPU times: user 17.4 ms, sys: 107 µs, total: 17.5 ms
Wall time: 17.5 ms


In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_24_try_1.pickle